In [6]:
import pandas as pd

# 1. 데이터 불러오기
print("데이터를 불러오는 중...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
layout_info = pd.read_csv('layout_info.csv')

# 2. layout_id를 기준으로 병합 (Left Join)
print("데이터 병합 중...")
train_merged = pd.merge(train, layout_info, on='layout_id', how='left')
test_merged = pd.merge(test, layout_info, on='layout_id', how='left')

# 3. 병합된 데이터 저장하기
print("병합된 파일 저장 중...")
# index=False를 설정해야 쓸데없는 인덱스 열(0, 1, 2...)이 추가로 저장되지 않습니다.
train_merged.to_csv('train_merged.csv', index=False)
test_merged.to_csv('test_merged.csv', index=False)

print("🎉 저장 완료!")
print(f"저장된 Train 파일 크기: {train_merged.shape}")
print(f"저장된 Test 파일 크기: {test_merged.shape}")

데이터를 불러오는 중...
데이터 병합 중...
병합된 파일 저장 중...
🎉 저장 완료!
저장된 Train 파일 크기: (250000, 108)
저장된 Test 파일 크기: (50000, 107)


In [7]:
# 결측치 기준을 정하자
import pandas as pd

# 방금 저장한 병합 데이터를 불러옵니다. (메모리에 이미 있다면 생략 가능합니다)
train = pd.read_csv('train_merged.csv')
test = pd.read_csv('test_merged.csv')

# 결측치가 1개 이상 있는 컬럼만 쏙 뽑아서 보여주는 함수
def show_missing_values(df, dataset_name):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    
    print(f"=== {dataset_name} 데이터 결측치 ===")
    if missing.empty:
        print("🎉 결측치가 하나도 없습니다!\n")
    else:
        # 결측치 개수와 비율(%)을 함께 출력
        missing_df = pd.DataFrame({
            '결측치 개수': missing,
            '비율(%)': (missing / len(df) * 100).round(2)
        })
        print(missing_df, "\n")

show_missing_values(train, "Train")
show_missing_values(test, "Test")

=== Train 데이터 결측치 ===
                       결측치 개수  비율(%)
avg_recovery_time       32529  13.01
congestion_score        32250  12.90
avg_charge_wait         30696  12.28
battery_mean            30320  12.13
charge_efficiency_pct   30052  12.02
...                       ...    ...
low_battery_ratio       29404  11.76
battery_std             29379  11.75
aisle_traffic_score     29359  11.74
task_reassign_15m       29337  11.73
air_quality_idx         29322  11.73

[86 rows x 2 columns] 

=== Test 데이터 결측치 ===
                         결측치 개수  비율(%)
avg_recovery_time          6676  13.35
congestion_score           6318  12.64
avg_charge_wait            6199  12.40
battery_cycle_count_avg    6135  12.27
wifi_signal_db             6090  12.18
...                         ...    ...
conveyor_speed_mps         5806  11.61
packaging_material_cost    5804  11.61
task_reassign_15m          5799  11.60
aisle_traffic_score        5764  11.53
warehouse_temp_avg         5739  11.48

[86 rows x 2 column

In [8]:
import pandas as pd

# 1. 병합된 데이터 불러오기 (이미 메모리에 있다면 생략 가능)
print("데이터를 불러오는 중...")
train = pd.read_csv('train_merged.csv')
test = pd.read_csv('test_merged.csv')

# 2. 결측치 처리 전략 세우기
# 타겟 변수(예측해야 할 정답)는 결측치 처리에서 제외합니다.
target_col = 'avg_delay_minutes_next_30m'

# 0으로 채울 컬럼들의 키워드 (횟수, 시간, 대기열, 점수, 비율 등)
zero_keywords = ['count', 'wait', 'time', 'score', 'queue', 'reassign', 'blocked', 'collision', 'fault', 'ratio', 'traffic']

# 컬럼 자동 분류
zero_cols = [col for col in train.columns if any(kw in col for kw in zero_keywords) and col != target_col]

# 수치형 컬럼 중 zero_cols에 해당하지 않는 나머지 (센서, 온도, 배터리 등) -> 중앙값으로 채움
numeric_cols = train.select_dtypes(include=['float64', 'int64']).columns
median_cols = [col for col in numeric_cols if col not in zero_cols and col != target_col]

print(f"0으로 채울 컬럼 수: {len(zero_cols)}개")
print(f"중앙값으로 채울 컬럼 수: {len(median_cols)}개")

# 3. 결측치 채우기
print("\n결측치를 채우는 중...")

# [Train 데이터 처리]
train[zero_cols] = train[zero_cols].fillna(0)
for col in median_cols:
    train[col] = train[col].fillna(train[col].median())

# [Test 데이터 처리] - 주의: 중앙값은 반드시 Train 데이터의 중앙값을 사용해야 합니다! (Data Leakage 방지)
test[zero_cols] = test[zero_cols].fillna(0)
for col in median_cols:
    test[col] = test[col].fillna(train[col].median())

# 혹시 모를 범주형 데이터(Object)의 결측치는 최빈값(가장 자주 나오는 값)으로 처리
cat_cols = train.select_dtypes(include=['object']).columns
for col in cat_cols:
    if col not in ['ID', 'layout_id', 'scenario_id']:  # 식별자 제외
        mode_val = train[col].mode()[0]
        train[col] = train[col].fillna(mode_val)
        test[col] = test[col].fillna(mode_val)

# 4. 결과 최종 확인
print("\n=== 처리 후 결측치 총합 ===")
print(f"Train 남은 결측치: {train.isnull().sum().sum()} 개")
print(f"Test 남은 결측치: {test.isnull().sum().sum()} 개")

# 5. 전처리 완료된 데이터 저장
train.to_csv('train_filled.csv', index=False)
test.to_csv('test_filled.csv', index=False)
print("\n🎉 'train_filled.csv' 및 'test_filled.csv'로 저장 완료!")

데이터를 불러오는 중...
0으로 채울 컬럼 수: 39개
중앙값으로 채울 컬럼 수: 64개

결측치를 채우는 중...


C:\Users\kyw41\AppData\Local\Temp\ipykernel_18792\359938258.py:39: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include=['object']).columns



=== 처리 후 결측치 총합 ===
Train 남은 결측치: 0 개
Test 남은 결측치: 0 개

🎉 'train_filled.csv' 및 'test_filled.csv'로 저장 완료!
